# CFPB Consumer Complaints - Exploratory Analysis

Notebook walking through the U.S. Consumer Financial Protection Bureau (CFPB) complaint database. The raw file is on Kaggle: https://www.kaggle.com/datasets/selener/consumer-complaint-database - about 1.28M complaints filed against ~5,275 companies between Dec 2011 and May 2019.

I'm using this dataset because every row is a "failure event" - the moment a consumer escalated to a federal regulator because a financial product didn't work for them. That's a high-signal dataset for risk, compliance, and CX teams, and it's big enough (~1.6 GB in memory) that it's a real test of a single-machine pandas pipeline.

### Questions I want to answer

1. Who attracts the most complaints, and how concentrated is the market?
2. Which products and issues are driving volume?
3. How has the mix changed over time? Any obvious external events?
4. What does resolution look like - and where does the system fail consumers?
5. Are there segments (company size, geography) with weaker operational performance?
6. What would I recommend doing about it?

Notes on style: I'm keeping intermediate inspections in here on purpose  feels more useful as a reference than a clean deck.

## 1. Setup

In [1]:
!pip install numpy pandas matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Sticking with a fixed palette so the charts feel consistent later on
NAVY, TEAL, CORAL, GOLD, SAGE, SLATE = "#1E2761", "#028090", "#F96167", "#F2A516", "#84B59F", "#475569"
GRID = "#E2E8F0"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.titleweight": "bold", "axes.titlesize": 13,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "font.size": 10, "figure.dpi": 100,
})

DATA_PATH = "/home/rocky/.cache/kagglehub/datasets/selener/consumer-complaint-database/versions/1/rows.csv"
print('Setup complete. Pandas:', pd.__version__, '| Numpy:', np.__version__)

Setup complete. Pandas: 2.3.3 | Numpy: 2.0.2


## 2. Load the data

In [3]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory footprint: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
df.head(3)

Shape: 1,282,355 rows × 18 columns
Memory footprint: 1,746.1 MB


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,05/10/2019,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,NAVY FEDERAL CREDIT UNION,FL,328XX,Older American,NaN,Web,05/10/2019,In progress,Yes,NaN,3238275
1,05/10/2019,Checking or savings account,Other banking product or service,Managing an account,Deposits and withdrawals,NaN,NaN,BOEING EMPLOYEES CREDIT UNION,WA,98204,NaN,NaN,Referral,05/10/2019,Closed with explanation,Yes,NaN,3238228
2,05/10/2019,Debt collection,Payday loan debt,Communication tactics,Frequent or repeated calls,NaN,NaN,CURO Intermediate Holdings,TX,751XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3237964


In [4]:
# Just want to see what dtypes pandas inferred  most should be object/str
print("Columns and dtypes:")
print(df.dtypes)

Columns and dtypes:
Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Consumer consent provided?      object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Consumer disputed?              object
Complaint ID                     int64
dtype: object


## 3. Profiling - what are we actually working with?

Before doing anything, three quick checks:

1. Where are the holes? (missing values per column)
2. Are there duplicates? (each complaint should have a unique ID)
3. Does the date range make sense?

In [5]:
# How much is missing in each column?
missing = df.isna().sum().to_frame("missing")
missing["pct_missing"] = (missing["missing"] / len(df) * 100).round(2)
missing.sort_values("pct_missing", ascending=False)

,missing,pct_missing
Tags,1106712,86.30
Consumer complaint narrative,898791,70.09
Company public response,833273,64.98
Consumer consent provided?,591701,46.14
Sub-issue,531186,41.42
Consumer disputed?,513854,40.07
Sub-product,235166,18.34
ZIP code,115298,8.99
State,19400,1.51
Date received,0,0.00


A few things worth flagging from that table:

- **Tags (86% missing)** and **Consumer complaint narrative (70% missing)** are opt-in - the consumer has to consent to share them. Not useful for headline metrics, but the ~30% with narratives is a goldmine for NLP work later.
- **Consumer disputed? (40% missing)** isn't random missingness - CFPB stopped collecting this field around April 2017. So any dispute-rate analysis has to be restricted to the pre-2017 population, with sample size disclosed.
- **Sub-product (18%)**, **ZIP code (9%)**, and **State (1.5%)** are manageable - I'll fill those with `Unknown` where they're used as group keys.
- The spine of the dataset - `Date received`, `Product`, `Company`, `Submitted via`, `Complaint ID` - is essentially complete. Good.

In [6]:
# Each complaint should have a unique ID
print("Duplicate Complaint IDs:", df['Complaint ID'].duplicated().sum())

# Parse dates so we can check the range
df['Date received'] = pd.to_datetime(df['Date received'], errors='coerce')
print("Date range:", df['Date received'].min().date(), "?", df['Date received'].max().date())
print("Invalid dates:", df['Date received'].isna().sum())

Duplicate Complaint IDs: 0
Date range: 2011-12-01 ? 2019-05-10
Invalid dates: 0


In [7]:
# How many distinct values in each categorical? Helps decide what's useful for group-bys.
cat_cols = ['Product','Sub-product','Issue','Sub-issue','Company','State',
            'Submitted via','Company response to consumer','Timely response?',
            'Consumer disputed?','Tags','Company public response']

cardinality = {col: df[col].nunique() for col in cat_cols}
pd.DataFrame.from_dict(cardinality, orient='index', columns=['unique_values']) \
  .sort_values('unique_values', ascending=False)

,unique_values
Company,5275
Sub-issue,218
Issue,167
Sub-product,76
State,63
Product,18
Company public response,10
Company response to consumer,8
Submitted via,6
Tags,3


## 4. Cleaning & feature engineering

Plan:

- Parse both date columns properly
- Fill `State`, `Submitted via`, `Company response to consumer` with `Unknown` so they survive group-bys
- Pull out time features (`year`, `month`, `year_month`, `dow`)
- Compute `lag_days` = days between received and sent - a rough operational responsiveness signal
- Create binary flags for the things I'll measure repeatedly: untimely response, disputed, relief outcomes
- Bucket companies into tiers (Top 10 / Top 11-100 / long tail) - closest thing to segmentation we can do without a customer ID

No rows dropped except those missing date or product - in practice that's zero, but the safety net stays.

In [8]:
# Dates
df['Date received']        = pd.to_datetime(df['Date received'], errors='coerce')
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'], errors='coerce')

before = len(df)
df = df.dropna(subset=['Date received', 'Product'])
print(f"Dropped {before - len(df)} rows missing date/product")

# Fill the categoricals that we'll group by
df['State']                        = df['State'].fillna('Unknown')
df['Submitted via']                = df['Submitted via'].fillna('Unknown')
df['Company response to consumer'] = df['Company response to consumer'].fillna('Unknown')

# Time features
df['year']       = df['Date received'].dt.year
df['month']      = df['Date received'].dt.month
df['year_month'] = df['Date received'].dt.to_period('M').astype(str)
df['dow']        = df['Date received'].dt.day_name()
df['lag_days']   = (df['Date sent to company'] - df['Date received']).dt.days

# Binary flags  easier to mean() these later than to keep filtering strings
df['is_untimely']      = (df['Timely response?'] == 'No').astype(int)
df['is_disputed']      = (df['Consumer disputed?'] == 'Yes').astype(int)
df['has_dispute_data'] = df['Consumer disputed?'].notna().astype(int)

RELIEF = {'Closed with monetary relief', 'Closed with non-monetary relief', 'Closed with relief'}
df['is_any_relief']       = df['Company response to consumer'].isin(RELIEF).astype(int)
df['is_monetary_relief']  = (df['Company response to consumer'] == 'Closed with monetary relief').astype(int)
df['is_explanation_only'] = (df['Company response to consumer'] == 'Closed with explanation').astype(int)

# Company tiers  Top 10, next 90, then everyone else
top_co_volume = df['Company'].value_counts()
TIER1 = top_co_volume.head(10).index
TIER2 = top_co_volume.iloc[10:100].index

def tier(c):
    if c in TIER1: return 'Tier 1 (Top 10)'
    if c in TIER2: return 'Tier 2 (Top 11-100)'
    return 'Tier 3 (Long tail)'

df['company_tier'] = df['Company'].map(tier)

print("Cleaning complete. Shape:", df.shape)
df[['year_month','company_tier','is_untimely','is_any_relief','lag_days']].head()

Dropped 0 rows missing date/product
Cleaning complete. Shape: (1282355, 30)


,year_month,company_tier,is_untimely,is_any_relief,lag_days
0,2019-05,Tier 2 (Top 11-100),0,0,0
1,2019-05,Tier 3 (Long tail),0,0,0
2,2019-05,Tier 3 (Long tail),0,0,0
3,2019-05,Tier 2 (Top 11-100),0,0,0
4,2019-05,Tier 2 (Top 11-100),0,0,0


In [9]:
# Quick sanity check - lag_days should mostly be 0-2 days. Anything weird?
print("Lag days summary:")
print(df['lag_days'].describe())

print("\nNegative lag_days (sent before received - data error):",
      (df['lag_days'] < 0).sum())
print("Lag > 30 days (potential processing backlog):",
      (df['lag_days'] > 30).sum())

Lag days summary:
count    1.282355e+06
mean     3.174895e+00
std      1.493060e+01
min     -1.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      1.962000e+03
Name: lag_days, dtype: float64

Negative lag_days (sent before received - data error): 7052
Lag > 30 days (potential processing backlog): 26177


#### A handful of rows have negative or extreme lag values - looks like data-entry quirks rather than anything meaningful. 
#### I'm flagging them but leaving them in for the descriptive analysis. 
#### If we were building an SLA forecasting model, these would need to come out.

## 5. Headline numbers

Before diving into individual insights - here are the numbers I'd want on a single slide. Everything that follows is a drill-down into one of these.

In [10]:
kpis = {
    'Total complaints'                      : f"{len(df):,}",
    'Unique companies'                      : f"{df['Company'].nunique():,}",
    'Unique products'                       : f"{df['Product'].nunique()}",
    'Date range'                            : f"{df['Date received'].min().date()} ? {df['Date received'].max().date()}",
    'Top 10 companies share'                : f"{df['Company'].value_counts().head(10).sum()/len(df)*100:.1f}%",
    'Top product (Mortgage) share'          : f"{(df['Product']=='Mortgage').mean()*100:.1f}%",
    'Web channel share'                     : f"{(df['Submitted via']=='Web').mean()*100:.1f}%",
    'Timely response rate'                  : f"{(df['Timely response?']=='Yes').mean()*100:.1f}%",
    'Any-relief outcome rate'               : f"{df['is_any_relief'].mean()*100:.1f}%",
    'Monetary-relief outcome rate'          : f"{df['is_monetary_relief'].mean()*100:.1f}%",
    'Explanation-only outcome rate'         : f"{df['is_explanation_only'].mean()*100:.1f}%",
    'Consumer dispute rate (pre-2017 pop.)' : f"{df.loc[df['has_dispute_data']==1,'is_disputed'].mean()*100:.1f}%",
}

pd.DataFrame.from_dict(kpis, orient='index', columns=['value'])

,value
Total complaints,"1,282,355"
Unique companies,"5,275"
Unique products,18
Date range,2011-12-01 ? 2019-05-10
Top 10 companies share,52.2%
Top product (Mortgage) share,21.7%
Web channel share,73.7%
Timely response rate,97.5%
Any-relief outcome rate,18.6%
Monetary-relief outcome rate,5.8%


## 6. Insight #1 - Market concentration is extreme

Out of 5,275 companies, the top 10 account for ~52% of all complaints. The three credit bureaus (Equifax, Experian, TransUnion) alone are about a quarter of the dataset.

In [11]:
top15 = df['Company'].value_counts().head(15)
share = top15 / len(df) * 100

result = pd.DataFrame({'complaints': top15, 'pct_of_total': share.round(2)})
result

,complaints,pct_of_total
Company,,
"EQUIFAX, INC.",115703,9.02
Experian Information Solutions Inc.,103784,8.09
"TRANSUNION INTERMEDIATE HOLDINGS, INC.",96587,7.53
"BANK OF AMERICA, NATIONAL ASSOCIATION",82104,6.40
WELLS FARGO & COMPANY,70919,5.53
JPMORGAN CHASE & CO.,60221,4.70
"CITIBANK, N.A.",49058,3.83
CAPITAL ONE FINANCIAL CORPORATION,34581,2.70
"Navient Solutions, LLC.",29296,2.28


In [12]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

NAVY_  = '#1E2761'
CORAL_ = '#F96167'

total = len(df)
top15 = df['Company'].value_counts().head(15)
top15_rev = top15.iloc[::-1]
top10_share = top15.head(10).sum() / total * 100
bureau_share = top15.head(3).sum() / total * 100

CREDIT_BUREAUS = {
    'EQUIFAX, INC.',
    'Experian Information Solutions Inc.',
    'TRANSUNION INTERMEDIATE HOLDINGS, INC.',
}

colors = [CORAL_ if name in CREDIT_BUREAUS else NAVY_ for name in top15_rev.index]

# Clean up labels - title case ALL-CAPS names, truncate the very long ones
labels = [c.title() if c.isupper() else c for c in top15_rev.index]
labels = [c if len(c) < 34 else c[:31] + '...' for c in labels]
full_names = list(top15_rev.index)

xmax = float(top15_rev.max()) * 1.30

fig = go.Figure()

fig.add_trace(go.Bar(
    y=labels,
    x=list(top15_rev.values),
    orientation='h',
    marker=dict(color=colors, line=dict(color='white', width=1.5)),
    text=[f'<b>{v/1000:,.0f}K</b>   {v/total*100:.1f}%' for v in top15_rev.values],
    textposition='outside',
    textfont=dict(size=11, color='#334155'),
    customdata=list(zip(full_names, [v/total*100 for v in top15_rev.values])),
    hovertemplate='<b>%{customdata[0]}</b><br>Complaints: <b>%{x:,}</b><br>Share: <b>%{customdata[1]:.2f}%</b><extra></extra>',
    cliponaxis=False,
))

# Subtle key in the subtitle line
key_html = (
    f'<span style="color:{CORAL_}">|</span> <span style="color:#475569">Credit bureaus</span>'
    f'   <span style="color:{NAVY_}">|</span> <span style="color:#475569">Other top 15</span>'
)

fig.update_layout(
    title=dict(
        text=(
            f'<b style="font-size:24px;color:#0F172A">Top 10 companies = {top10_share:.0f}% of all complaints</b><br>'
            f'<span style="font-size:13px;color:#64748B">'
            f'Out of {df["Company"].nunique():,} companies in the dataset, the three credit bureaus alone '
            f'account for <b>{bureau_share:.1f}%</b>'
            '</span><br>'
            f'<span style="font-size:11px">{key_html}</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    margin=dict(l=270, r=110, t=150, b=90),
    xaxis=dict(
        title=dict(
            text=(
                '<i style="color:#94A3B8;font-size:10px">'
                'Source: CFPB Consumer Complaint Database  |  '
                f'n = {total:,} complaints  |  Dec 2011 to May 2019'
                '</i>'
            ),
            standoff=25,
        ),
        range=[0, xmax],
        showgrid=True, gridcolor='#EEF2F6', griddash='dot',
        showline=False, ticks='', zeroline=False,
        tickfont=dict(size=10, color='#94A3B8'),
        tickformat=',.0f',
    ),
    yaxis=dict(
        title='',
        showgrid=False, showline=False, ticks='',
        tickfont=dict(size=11, color='#334155'),
        automargin=True,
    ),
    showlegend=False,
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.30,
)

fig.show()

**So what?** A regulator auditing just the top 10 companies covers more than half of all consumer harm signals in the entire dataset. For any individual bank, this means complaint-share is essentially a public benchmark - you're being directly compared against ~10 peers every day on a federal website. Moving the needle by even 1pp is ~13K fewer complaints.

The long tail (Tier 3 in our segmentation) gets attention later - it turns out the outcomes are noticeably worse there.

## 7. Insight #2 - Mortgage and Credit Reporting are trading places

Mortgage used to be the dominant complaint category - about half of all volume in 2012. By 2019 it's down to ~9%. Meanwhile "Credit reporting, credit repair services, or other personal consumer reports" barely existed as a category before 2017, and now it's half of yearly volume. Two forces driving this: the post-2008 mortgage crisis tail receding, and the Equifax breach (Sept 2017) pushing credit-data accuracy into public awareness.

Let's see the time series first, then the product mix shift.

In [13]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

NAVY_  = '#1E2761'
CORAL_ = '#F96167'

monthly = df.groupby('year_month').size().reset_index(name='count')
monthly['dt'] = pd.to_datetime(monthly['year_month'])
monthly['rolling12'] = monthly['count'].rolling(12, min_periods=3).mean()

peak = monthly.loc[monthly['count'].idxmax()]

# Headline numbers for the subtitle (plain ASCII only)
first_year_avg = monthly['count'].head(12).mean()
last_year_avg  = monthly['count'].tail(12).mean()
multiple = last_year_avg / first_year_avg if first_year_avg > 0 else 0

fig = go.Figure()

# Soft fill under the monthly line
fig.add_trace(go.Scatter(
    x=monthly['dt'], y=monthly['count'],
    mode='lines',
    line=dict(color=NAVY_, width=0),
    fill='tozeroy',
    fillcolor='rgba(30, 39, 97, 0.10)',
    hoverinfo='skip',
    showlegend=False,
))

# Monthly volume line
fig.add_trace(go.Scatter(
    x=monthly['dt'], y=monthly['count'],
    mode='lines',
    line=dict(color=NAVY_, width=1.6),
    name='Monthly volume',
    hovertemplate='<b>%{x|%b %Y}</b><br>Volume: <b>%{y:,}</b><extra></extra>',
))

# 12-month rolling average
fig.add_trace(go.Scatter(
    x=monthly['dt'], y=monthly['rolling12'],
    mode='lines',
    line=dict(color=CORAL_, width=2.6, dash='dash'),
    name='12-month rolling avg',
    hovertemplate='<b>%{x|%b %Y}</b><br>12-mo avg: <b>%{y:,.0f}</b><extra></extra>',
))

# Marker on the Equifax peak
fig.add_trace(go.Scatter(
    x=[peak['dt']], y=[peak['count']],
    mode='markers',
    marker=dict(size=11, color=CORAL_, line=dict(color='white', width=2)),
    hovertemplate=f"<b>Equifax breach</b><br>{peak['year_month']}<br>Volume: <b>{int(peak['count']):,}</b><extra></extra>",
    showlegend=False,
))

# Annotation for the Equifax spike
fig.add_annotation(
    x=peak['dt'], y=peak['count'],
    text=f"<b>Equifax breach</b><br>{peak['year_month']}  |  {int(peak['count']):,}",
    showarrow=True,
    arrowhead=2, arrowsize=1.1, arrowwidth=1.5, arrowcolor=CORAL_,
    ax=70, ay=-50,
    font=dict(color=CORAL_, size=11),
    bgcolor='rgba(255,255,255,0.95)',
    bordercolor=CORAL_, borderwidth=1.2, borderpad=6,
    align='left',
)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">Complaint volume has roughly tripled over seven years</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'Monthly intake, Dec 2011 to May 2019  |  '
            f'first-year avg <b>{first_year_avg/1000:.1f}K/mo</b> grew to <b>{last_year_avg/1000:.1f}K/mo</b> '
            f'(roughly <b>{multiple:.1f}x</b>)'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.96,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=560,
    margin=dict(l=70, r=40, t=120, b=100),
    xaxis=dict(
        title=dict(
            text=(
                '<i style="color:#94A3B8;font-size:10px">'
                'Source: CFPB Consumer Complaint Database  |  '
                'Solid line: raw monthly intake  |  '
                'Dashed line: 12-month rolling average'
                '</i>'
            ),
            standoff=25,
        ),
        showgrid=False, showline=False, ticks='',
        tickfont=dict(size=11, color='#334155'),
        showspikes=True, spikethickness=1, spikecolor='#CBD5E1',
        spikemode='across', spikesnap='cursor',
    ),
    yaxis=dict(
        title='',
        gridcolor='#EEF2F6', griddash='dot',
        showline=False, ticks='', zeroline=False,
        tickfont=dict(size=10.5, color='#94A3B8'),
        tickformat=',.0f',
    ),
    legend=dict(
        orientation='h', yanchor='top', y=1.04,
        xanchor='right', x=1.0,
        font=dict(size=11, color='#475569'),
        bgcolor='rgba(0,0,0,0)',
    ),
    hovermode='x unified',
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
)

fig.show()

In [14]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

heat = df.groupby(['Product', 'year']).size().unstack(fill_value=0)
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).head(10).index]
heat_pct = heat.div(heat.sum(axis=0), axis=1) * 100

y_labels = [p if len(p) < 42 else p[:39] + '...' for p in heat_pct.index]
x_labels = [str(int(y)) for y in heat_pct.columns]
full_names = list(heat_pct.index)
customdata = [[full_names[i] for _ in heat_pct.columns] for i in range(len(full_names))]

# Per-cell text overlay: white text on dark cells, dark text on light cells
DARK_CELL_THRESHOLD = 25
text_x, text_y, text_vals, text_colors = [], [], [], []
for i, prod in enumerate(y_labels):
    for j, yr in enumerate(x_labels):
        v = heat_pct.values[i, j]
        if v > 0.5:
            text_x.append(yr)
            text_y.append(prod)
            text_vals.append(f'<b>{v:.0f}%</b>')
            text_colors.append('white' if v > DARK_CELL_THRESHOLD else '#0F172A')

fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=heat_pct.values,
    x=x_labels,
    y=y_labels,
    customdata=customdata,
    hovertemplate='<b>%{customdata}</b><br>Year: %{x}<br>Share of year: <b>%{z:.1f}%</b><extra></extra>',
    colorscale='YlGnBu',
    zmin=0, zmax=max(60, heat_pct.values.max()),
    xgap=2, ygap=2,
    colorbar=dict(
        title=dict(text='Share<br>of year', font=dict(size=10, color='#475569'), side='right'),
        thickness=14, len=0.7, ticksuffix='%',
        outlinewidth=0, tickfont=dict(size=10, color='#94A3B8'),
    ),
    showscale=True,
    hoverongaps=False,
))

fig.add_trace(go.Scatter(
    x=text_x, y=text_y,
    mode='text',
    text=text_vals,
    textfont=dict(size=11, color=text_colors),
    hoverinfo='skip',
    showlegend=False,
))

# Subtitle stats - plain ASCII only
mort_2012 = heat_pct.loc['Mortgage', 2012] if ('Mortgage' in heat_pct.index and 2012 in heat_pct.columns) else None
mort_last = heat_pct.loc['Mortgage'].iloc[-1] if 'Mortgage' in heat_pct.index else None
cr_label  = next((p for p in heat_pct.index if 'credit reporting' in p.lower()), None)
cr_2019   = heat_pct.loc[cr_label].iloc[-1] if cr_label else None

subtitle_bits = ['Share of yearly complaints (%), top 10 products']
if mort_2012 is not None and mort_last is not None:
    subtitle_bits.append(f'Mortgage: <b>{mort_2012:.0f}%</b> to <b>{mort_last:.0f}%</b> ({x_labels[1]}-{x_labels[-1]})')
if cr_2019 is not None:
    subtitle_bits.append(f'Credit reporting: now <b>{cr_2019:.0f}%</b> of yearly volume')
subtitle = '  |  '.join(subtitle_bits)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">Product mix has flipped: mortgage out, credit reporting in</b><br>'
            f'<span style="font-size:13px;color:#64748B">{subtitle}</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=620,
    margin=dict(l=290, r=120, t=120, b=100),
    xaxis=dict(
        title=dict(
            text=(
                '<i style="color:#94A3B8;font-size:10px">'
                'Source: CFPB Consumer Complaint Database  |  '
                'Columns sum to 100% each year  |  '
                'Cells under 0.5% left blank for legibility'
                '</i>'
            ),
            standoff=25,
        ),
        side='bottom',
        showgrid=False, showline=False, ticks='',
        tickfont=dict(size=11, color='#334155'),
    ),
    yaxis=dict(
        title='',
        autorange='reversed',
        showgrid=False, showline=False, ticks='',
        tickfont=dict(size=10.5, color='#334155'),
        automargin=True,
    ),
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    showlegend=False,
)

fig.show()

**So what?** The "next mortgage" is already here - credit-reporting accuracy is the dominant friction point now, and any institution touching consumer credit data is in the regulator's crosshairs.

Two practical implications: (1) capacity-planning models built on 2014-2016 data will materially under-forecast credit-reporting volume, and (2) the product taxonomy itself shifts - "Credit reporting" was renamed to the longer "Credit reporting, credit repair services, -" mid-2017. Worth treating taxonomy as a dimension table downstream so renames don't break pipelines.

## 8. Insight #3 - Operational quality varies sharply by company tier

Long-tail companies (Tier 3) miss the response deadline ~9.5% of the time, vs about 1% for the top tiers. They're also roughly half as likely to grant any form of relief. The headline KPI (97.5% timely overall) hides a real quality gap underneath.

In [15]:
tier_metrics = df.groupby('company_tier').agg(
    complaints=('Complaint ID', 'count'),
    untimely_rate=('is_untimely', 'mean'),
    any_relief=('is_any_relief', 'mean'),
    explanation_only=('is_explanation_only', 'mean'),
).reindex(['Tier 1 (Top 10)', 'Tier 2 (Top 11-100)', 'Tier 3 (Long tail)'])

tier_metrics_pct = (tier_metrics[['untimely_rate', 'any_relief', 'explanation_only']] * 100).round(1)
tier_metrics_pct.insert(0, 'complaints', tier_metrics['complaints'])
tier_metrics_pct

,complaints,untimely_rate,any_relief,explanation_only
company_tier,,,,
Tier 1 (Top 10),670003,1.1,21.4,75.0
Tier 2 (Top 11-100),384418,0.7,18.6,78.3
Tier 3 (Long tail),227934,9.5,10.3,83.2


In [16]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

NAVY_  = '#1E2761'
CORAL_ = '#F96167'
SAGE_  = '#84B59F'

tm = tier_metrics_pct.reset_index()
tiers = list(tm['company_tier'])

# Key takeaway numbers for the subtitle
untimely_t3 = tm.loc[tm['company_tier'] == 'Tier 3 (Long tail)', 'untimely_rate'].iloc[0]
untimely_t1 = tm.loc[tm['company_tier'] == 'Tier 1 (Top 10)',   'untimely_rate'].iloc[0]
relief_t1   = tm.loc[tm['company_tier'] == 'Tier 1 (Top 10)',   'any_relief'].iloc[0]
relief_t3   = tm.loc[tm['company_tier'] == 'Tier 3 (Long tail)', 'any_relief'].iloc[0]

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Untimely',
    x=tiers, y=tm['untimely_rate'],
    marker=dict(color=CORAL_, line=dict(color='white', width=1.5)),
    text=[f'<b>{v:.1f}%</b>' for v in tm['untimely_rate']],
    textposition='outside',
    textfont=dict(size=11, color=CORAL_),
    hovertemplate='<b>%{x}</b><br>Untimely: <b>%{y:.1f}%</b><extra></extra>',
    cliponaxis=False,
))
fig.add_trace(go.Bar(
    name='Explanation only',
    x=tiers, y=tm['explanation_only'],
    marker=dict(color=NAVY_, line=dict(color='white', width=1.5)),
    text=[f'<b>{v:.1f}%</b>' for v in tm['explanation_only']],
    textposition='outside',
    textfont=dict(size=11, color=NAVY_),
    hovertemplate='<b>%{x}</b><br>Explanation only: <b>%{y:.1f}%</b><extra></extra>',
    cliponaxis=False,
))
fig.add_trace(go.Bar(
    name='Any relief',
    x=tiers, y=tm['any_relief'],
    marker=dict(color=SAGE_, line=dict(color='white', width=1.5)),
    text=[f'<b>{v:.1f}%</b>' for v in tm['any_relief']],
    textposition='outside',
    textfont=dict(size=11, color=SAGE_),
    hovertemplate='<b>%{x}</b><br>Any relief: <b>%{y:.1f}%</b><extra></extra>',
    cliponaxis=False,
))

key_html = (
    f'<span style="color:{CORAL_}">¦</span> untimely   '
    f'<span style="color:{NAVY_}">¦</span> explanation only   '
    f'<span style="color:{SAGE_}">¦</span> any relief'
)

ymax = float(max(tm['explanation_only'])) * 1.22

fig.update_layout(
    barmode='group',
    bargap=0.32,
    bargroupgap=0.08,
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">Long-tail companies are noticeably worse on every operational metric</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'Outcome rates by company tier · Tier 3 is untimely <b>{untimely_t3:.1f}%</b> of the time vs '
            f'<b>{untimely_t1:.1f}%</b> for Tier 1, and delivers relief <b>{relief_t3:.1f}%</b> vs <b>{relief_t1:.1f}%</b>'
            '</span><br>'
            f'<span style="font-size:11px">{key_html}</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=560,
    margin=dict(l=60, r=40, t=150, b=100),
    xaxis=dict(
        title=dict(
            text=(
                '<i style="color:#94A3B8;font-size:10px">'
                'Source: CFPB Consumer Complaint Database  ·  '
                'Tier 1 = Top 10 companies by volume, Tier 2 = ranks 11-100, Tier 3 = all others'
                '</i>'
            ),
            standoff=25,
        ),
        showgrid=False, showline=False, ticks='',
        tickfont=dict(size=12, color='#334155'),
    ),
    yaxis=dict(
        title='',
        range=[0, ymax],
        ticksuffix='%',
        gridcolor='#EEF2F6', griddash='dot',
        showline=False, ticks='', zeroline=False,
        tickfont=dict(size=10.5, color='#94A3B8'),
    ),
    showlegend=False,
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
)

fig.show()

**So what?** A targeted compliance programme aimed at Tier 3 would lift the system-wide untimely rate well below 1%. And lifting Tier 3 relief rates by even 5pp toward the Tier 1 average would translate to ~50,000 additional consumer remediations across the period. Mid-market players hoping to scale into Tier 2 can see exactly what quality gap they need to close.

## 9. Insight #4 - Resolution outcomes are mostly explanatory, not remedial

77.5% of complaints close with explanation only. Only 5.8% deliver monetary relief, and 18.6% deliver any form of relief at all. So the system is fast (97.5% timely) but the outcomes are soft.

In [17]:
resp = df['Company response to consumer'].value_counts().to_frame('count')
resp['pct'] = (resp['count'] / len(df) * 100).round(2)
resp

,count,pct
Company response to consumer,,
Closed with explanation,993221,77.45
Closed with non-monetary relief,158716,12.38
Closed with monetary relief,74243,5.79
Closed without relief,17868,1.39
Closed,17611,1.37
In progress,9277,0.72
Untimely response,6108,0.48
Closed with relief,5304,0.41
Unknown,7,0.00


In [18]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

NAVY_  = '#1E2761'
TEAL_  = '#028090'
CORAL_ = '#F96167'
SAGE_  = '#84B59F'

resp_plot = resp['count'].iloc[::-1]
total_n = len(df)

def color_for(label):
    l = label.lower()
    if 'monetary' in l or label == 'Closed with relief':
        return SAGE_
    if label == 'Closed with non-monetary relief':
        return TEAL_
    if label == 'Closed with explanation':
        return NAVY_
    if 'without' in l or label == 'Untimely response':
        return CORAL_
    return '#94A3B8'

colors = [color_for(l) for l in resp_plot.index]

# Headline numbers for the subtitle
expl_pct = (df['Company response to consumer'] == 'Closed with explanation').mean() * 100
relief_pct = df['is_any_relief'].mean() * 100

xmax = resp_plot.max() * 1.32

fig = go.Figure()

fig.add_trace(go.Bar(
    y=list(resp_plot.index),
    x=list(resp_plot.values),
    orientation='h',
    marker=dict(color=colors, line=dict(color='white', width=1.5)),
    text=[f'<b>{v/1000:,.0f}K</b>   {v/total_n*100:.1f}%' for v in resp_plot.values],
    textposition='outside',
    textfont=dict(size=11, color='#334155'),
    customdata=[v/total_n*100 for v in resp_plot.values],
    hovertemplate='<b>%{y}</b><br>Count: %{x:,}<br>Share: %{customdata:.2f}%<extra></extra>',
    cliponaxis=False,
))

key_html = (
    f'<span style="color:{SAGE_}">¦</span> monetary/any relief   '
    f'<span style="color:{TEAL_}">¦</span> non-monetary relief   '
    f'<span style="color:{NAVY_}">¦</span> explanation only   '
    f'<span style="color:{CORAL_}">¦</span> no relief / untimely'
)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">Fast, but rarely remedial</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'How complaints close · {expl_pct:.1f}% close with explanation only, '
            f'only {relief_pct:.1f}% deliver any form of relief'
            '</span><br>'
            f'<span style="font-size:11px">{key_html}</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=600,
    margin=dict(l=270, r=80, t=150, b=90),
    xaxis=dict(
        title=dict(
            text=(
                '<i style="color:#94A3B8;font-size:10px">'
                'Source: CFPB Consumer Complaint Database  ·  '
                f'n = {total_n:,} complaints  ·  Dec 2011 - May 2019'
                '</i>'
            ),
            standoff=25,
        ),
        range=[0, xmax],
        showgrid=True, gridcolor='#EEF2F6', griddash='dot',
        showline=False, ticks='', zeroline=False,
        tickfont=dict(size=10, color='#94A3B8'),
        tickformat=',.0f',
    ),
    yaxis=dict(
        title='',
        showgrid=False, showline=False, ticks='',
        tickfont=dict(size=11, color='#334155'),
        automargin=True,
    ),
    showlegend=False,
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.30,
)

fig.show()

**So what?** "Closed with explanation" looks like the cheap default - but the next insight shows that about 1 in 5 of those consumers come back disputing the resolution. That's not a free outcome.

There's also a missed-opportunity angle for ML: with 30% of complaints carrying a consumer narrative, a classifier could plausibly flag which "explanation-only" candidates are actually relief-eligible. A 1pp shift toward relief is ~13K additional remediated consumers.

## 10. Insight #5 - Mortgage, Consumer Loan, and Credit Card have above-average dispute rates

Caveat first: dispute data only exists for the pre-April-2017 portion of the dataset (CFPB stopped collecting it). So we're working with ~770K rows here, not the full 1.28M.

Within that population, the average dispute rate is 19.3%. Mortgage runs at 22.6%, Consumer Loan at 21.4%, Credit Card at 20.4% - meaning roughly 1 in 5 consumers who got an "explanation" came back unsatisfied. Worth knowing where that's concentrated.

In [19]:
disp = df[df['has_dispute_data'] == 1].groupby('Product').agg(
    n=('Complaint ID', 'count'),
    dispute_rate=('is_disputed', 'mean')
).reset_index()

# Filter out products with too few rows for the rate to be meaningful
disp = disp[disp['n'] >= 1000].sort_values('dispute_rate', ascending=False)
disp['dispute_rate_pct'] = (disp['dispute_rate'] * 100).round(2)

disp[['Product', 'n', 'dispute_rate_pct']]

,Product,n,dispute_rate_pct
7,Mortgage,226899,22.64
2,Consumer Loan,31605,21.42
3,Credit card,89190,20.41
8,Other financial service,1059,18.79
0,Bank account or service,86206,18.59
11,Student loan,32537,18.24
5,Debt collection,145835,17.58
4,Credit reporting,140432,15.75
6,Money transfers,5354,14.61
9,Payday loan,5544,14.38


In [20]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

avg_disp = df.loc[df['has_dispute_data'] == 1, 'is_disputed'].mean() * 100
disp_plot = disp.sort_values('dispute_rate_pct').tail(12).reset_index(drop=True)

NAVY_ = '#1E2761'
CORAL_ = '#F96167'

colors = [CORAL_ if r > avg_disp else NAVY_ for r in disp_plot['dispute_rate_pct']]
short_labels = [p if len(p) < 42 else p[:39] + '' for p in disp_plot['Product']]
xmax = disp_plot['dispute_rate_pct'].max() * 1.42

fig = go.Figure()

fig.add_vrect(
    x0=avg_disp, x1=xmax,
    fillcolor=CORAL_, opacity=0.06,
    layer='below', line_width=0,
)

fig.add_trace(go.Bar(
    y=short_labels,
    x=disp_plot['dispute_rate_pct'],
    orientation='h',
    marker=dict(color=colors, line=dict(color='white', width=1.5)),
    text=[f'<b>{v:.1f}%</b>   n={n:,}' for v, n in zip(disp_plot['dispute_rate_pct'], disp_plot['n'])],
    textposition='outside',
    textfont=dict(size=11, color='#334155'),
    customdata=list(zip(disp_plot['Product'], disp_plot['n'])),
    hovertemplate='<b>%{customdata[0]}</b><br>Dispute rate: <b>%{x:.1f}%</b><br>Sample size: %{customdata[1]:,}<extra></extra>',
    cliponaxis=False,
))

fig.add_shape(
    type='line',
    xref='x', yref='paper',
    x0=avg_disp, x1=avg_disp, y0=0, y1=1,
    line=dict(color='#475569', width=1.5, dash='dash'),
)

fig.add_annotation(
    x=avg_disp, y=1.0,
    xref='x', yref='paper',
    text=f'<b>avg {avg_disp:.1f}%</b>',
    showarrow=False,
    yanchor='bottom', xanchor='center',
    font=dict(color='#475569', size=11),
    bgcolor='white', borderpad=3,
)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">About 1 in 5 consumers dispute the resolution</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'Dispute rate by product · pre-April-2017 population (n = {df["has_dispute_data"].sum():,})  ·  '
            f'<span style="color:{CORAL_}">¦</span> above avg   '
            f'<span style="color:{NAVY_}">¦</span> below avg'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=640,
    margin=dict(l=300, r=80, t=120, b=110),
    xaxis=dict(
        title=dict(
            text=(
                '<i style="color:#94A3B8;font-size:10px">'
                'Source: CFPB Consumer Complaint Database  ·  '
                'Products with fewer than 1,000 complaints excluded  ·  '
                'Dispute field discontinued by CFPB in April 2017'
                '</i>'
            ),
            standoff=25,
        ),
        range=[0, xmax],
        ticksuffix='%',
        gridcolor='#EEF2F6', griddash='dot',
        showline=False, ticks='', zeroline=False,
        tickfont=dict(size=10.5, color='#94A3B8'),
    ),
    yaxis=dict(
        title='',
        showgrid=False, showline=False, ticks='',
        tickfont=dict(size=11, color='#334155'),
        automargin=True,
    ),
    showlegend=False,
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.32,
)

fig.show()

**So what?** Mortgage portfolios should treat post-resolution dispute rate as a leading indicator of litigation exposure - it's both the largest product *and* the highest dispute rate. For consumer-loan and credit-card businesses, the 24pp gap to the population average represents thousands of preventable disputes over the period studied.

Operational caveat: since CFPB discontinued the field, any institution that wants this signal going forward has to recreate it internally - it's no longer in the public dataset.

## 11. Insight #6 - Channel mix is overwhelmingly digital, and accelerating

73.7% of complaints come in via the web. That share has grown from under 40% in 2012 to over 85% by 2019. Phone, fax, mail, and email are effectively residual at this point.

In [21]:
ch = df.groupby(['year', 'Submitted via']).size().unstack(fill_value=0)
ch_pct = ch.div(ch.sum(axis=1), axis=0) * 100

order = ['Web', 'Referral', 'Phone', 'Postal mail', 'Fax', 'Email', 'Unknown']
order = [c for c in order if c in ch_pct.columns]
ch_pct = ch_pct[order]

ch_pct.round(1)

Submitted via,Web,Referral,Phone,Postal mail,Fax,Email
year,,,,,,
2011,57.4,31.3,8.6,1.9,0.4,0.4
2012,44.8,38.6,8.7,6.3,1.3,0.2
2013,55.8,26.0,8.8,7.4,1.9,0.1
2014,69.0,15.4,6.7,7.4,1.5,0.0
2015,74.2,12.8,6.1,5.6,1.3,0.0
2016,73.5,12.5,6.4,6.2,1.4,0.0
2017,81.5,8.1,4.6,4.6,1.1,0.0
2018,81.2,8.6,4.6,3.5,2.1,0.0
2019,84.9,6.5,5.4,2.4,0.8,0.0


In [22]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

# Make sure years are strings and in order
years = [str(int(y)) for y in ch_pct.index]

# Stack order: Web at bottom (anchors the eye), rest above in volume order
stack_order = [c for c in ['Web','Referral','Phone','Postal mail','Email','Fax','Unknown']
               if c in ch_pct.columns]

# Web saturated, everything else greyed out
channel_colors = {
    'Web':         '#1E2761',
    'Referral':    '#94A3B8',
    'Phone':       '#B8C0CC',
    'Postal mail': '#CBD5E1',
    'Email':       '#DCE3EC',
    'Fax':         '#E5EAF1',
    'Unknown':     '#EEF2F6',
}

web_start = ch_pct['Web'].iloc[0]
web_end   = ch_pct['Web'].iloc[-1]

fig = go.Figure()

for ch in stack_order:
    vals = ch_pct[ch].values
    is_web = (ch == 'Web')
    fig.add_trace(go.Bar(
        x=years,
        y=vals,
        name=ch,
        marker=dict(color=channel_colors[ch], line=dict(color='white', width=2)),
        text=[f'<b>{v:.0f}%</b>' if is_web else '' for v in vals],
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(color='white', size=14),
        hovertemplate=f'<b>{ch}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>',
    ))

fig.update_layout(
    barmode='stack',
    bargap=0.30,
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">The web is now essentially the only channel</b><br>'
            f'<span style="font-size:13px;color:#64748B">Web share climbed from '
            f'<b style="color:#1E2761">{web_start:.0f}%</b> in {years[0]} to '
            f'<b style="color:#1E2761">{web_end:.0f}%</b> in {years[-1]}</span>'
        ),
        x=0.02, xanchor='left', y=0.95,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=560,
    margin=dict(l=70, r=40, t=110, b=110),
    xaxis=dict(
        type='category',
        categoryorder='array',
        categoryarray=years,
        title='',
        showgrid=False,
        showline=True, linecolor='#E2E8F0',
        ticks='',
        tickfont=dict(size=12, color='#334155'),
    ),
    yaxis=dict(
        title='', range=[0, 100], ticksuffix='%',
        gridcolor='#EEF2F6', griddash='dot',
        showline=False, ticks='', zeroline=False,
        tickfont=dict(size=10.5, color='#94A3B8'),
    ),
    legend=dict(
        orientation='h', yanchor='top', y=-0.12,
        xanchor='center', x=0.5,
        font=dict(size=11, color='#475569'),
        bgcolor='rgba(0,0,0,0)',
    ),
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
)

# Footer
fig.add_annotation(
    x=0, y=-0.26, xref='paper', yref='paper',
    text='<i style="color:#94A3B8;font-size:10px">Source: CFPB Consumer Complaint Database · Stacks total 100% per year</i>',
    showarrow=False, xanchor='left',
)

fig.show()

**So what?** The web intake form is effectively the whole customer experience now - its UX, categorization, and triage logic *are* the complaint process. That also means an automated ML triage layer would cover ~85% of incoming volume with a single integration point.

Worth keeping in mind: phone and mail still carry ~7% combined and disproportionately serve older and lower-income consumers (visible in the `Tags` field for the subset that filled it in). Don't let "almost all web" become "web only."

## 12. Geographic lens - per-capita tells a different story

Raw counts naturally favor the big states (CA, TX, FL). To see where complaints are *unusually* high, we need to normalize by population. Using 2017 Census estimates as the denominator - close enough to the dataset midpoint.

In [23]:
# 2017 Census population estimates
POP_2017 = {
    'CA':39536653,'TX':28304596,'FL':20984400,'NY':19849399,'PA':12805537,'IL':12802023,'OH':11658609,
    'GA':10429379,'NC':10273419,'MI':9962311,'NJ':9005644,'VA':8470020,'WA':7405743,'AZ':7016270,
    'MA':6859819,'TN':6715984,'IN':6666818,'MO':6113532,'MD':6052177,'WI':5795483,'CO':5607154,
    'MN':5576606,'SC':5024369,'AL':4874747,'LA':4684333,'KY':4454189,'OR':4142776,'OK':3930864,
    'CT':3588184,'UT':3101833,'IA':3145711,'NV':2998039,'AR':3004279,'MS':2984100,'KS':2913123,
    'NM':2088070,'NE':1920076,'WV':1815857,'ID':1716943,'HI':1427538,'NH':1342795,'ME':1335907,
    'MT':1050493,'RI':1059639,'DE':961939,'SD':869666,'ND':755393,'AK':739795,'DC':693972,
    'VT':623657,'WY':579315,'PR':3337177,
}

st = df.loc[df['State'] != 'Unknown', 'State'].value_counts().reset_index()
st.columns = ['State', 'complaints']
st['pop'] = st['State'].map(POP_2017)
st = st.dropna()
st['per_100k'] = (st['complaints'] / st['pop'] * 100000).round(1)

top_pc = st.sort_values('per_100k', ascending=False).head(15)
top_pc

,State,complaints,pop,per_100k
32,DC,6886,693972.0,992.3
4,GA,67102,10429379.0,643.4
34,DE,6150,961939.0,639.3
11,MD,36545,6052177.0,603.8
1,FL,126487,20984400.0,602.8
20,NV,16006,2998039.0,533.9
6,NJ,47799,9005644.0,530.8
0,CA,176009,39536653.0,445.2
10,VA,37471,8470020.0,442.4
3,NY,86143,19849399.0,434.0


In [24]:
!pip install plotly

Defaulting to user installation because normal site-packages is not writeable


In [25]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'iframe'   # forces inline render in Jupyter

# Full per-capita ranking (all states) - map should show everyone, not just top 15
map_df = st.sort_values('per_100k', ascending=False).copy()

# Drop PR - plotly's USA-states scope doesn't include it
map_df = map_df[map_df['State'] != 'PR']

fig = px.choropleth(
    map_df,
    locations='State',
    locationmode='USA-states',
    color='per_100k',
    scope='usa',
    color_continuous_scale='YlOrRd',
    labels={'per_100k': 'Per 100k'},
    hover_data={'State': True, 'complaints': ':,', 'pop': ':,', 'per_100k': ':.1f'},
)

fig.update_layout(
    title=dict(
        text='<b>Complaints per 100,000 residents</b><br>'
             '<span style="font-size:13px;color:#475569">Small finance-heavy states lead - DC, DE, MD top the ranking</span>',
        x=0.02, xanchor='left', y=0.96,
    ),
    geo=dict(bgcolor='white', lakecolor='white'),
    margin=dict(l=10, r=10, t=90, b=10),
    height=520,
    coloraxis_colorbar=dict(title='per 100k', thickness=14, len=0.7),
)

fig.show()

**Reading this:** DC, Delaware, and Maryland top the list - all small jurisdictions with heavy financial-sector concentration and strong consumer-protection awareness. That's partly genuine financial-services density and partly a "complaint-savvy population" effect.

The interesting cells are Florida, Georgia, and Nevada - they show up high on *both* per-capita and absolute counts. That combination (real volume + real propensity to complain) makes them the highest signal-to-noise targets for a compliance or CX investment.

## 13. Limitations & validation

Worth being honest about what this dataset can and can't tell us:

| Limitation | What it means | What I did about it |
|---|---|---|
| Self-selected dataset | Not every consumer issue becomes a CFPB complaint | Treat metrics as *complaint propensity*, not absolute consumer harm |
| Dispute field discontinued (Apr 2017) | 40% of rows have no dispute value | Compute dispute rates only on the pre-2017 population; disclose sample size |
| No customer identifier | Can't do RFM- or LTV-style analysis | Use company-tier segmentation as the closest proxy |
| No financial-impact field | Can't compute dollar value of relief | Use relief-rate as a proxy; flag as an enrichment opportunity |
| Single-machine pandas | 1.6 GB working set is near the comfort limit | Production version would need Spark/Databricks |
| Taxonomy drift over the period | "Credit reporting" was renamed mid-2017 | Treat both old and new labels; acknowledge the regime shift |

**Validation checks I ran:**

- Total complaints reconcile to CFPB published yearly totals (within rounding)
- Top company list cross-checks with CFPB annual reports
- The September 2017 spike aligns with the publicly documented Equifax breach announcement (Sept 7, 2017)  a strong external anchor for the time series

## 14. Recommendations

### Strategy
1. **Make credit-reporting accuracy the priority category.** It's overtaken mortgage as the dominant complaint driver - and it's not slowing down.
2. **Targeted Tier-3 supervision programme.** Long-tail companies miss SLAs at roughly 10× the rate of Top-100s. A focused compliance push there would pull the system-wide untimely rate below 1%.

### Operations
3. **Resolution-quality scorecard.** Track relief-rate (not just timely-response rate) at the company level. The system is fast at 97.5% timely but soft - only 5.8% deliver monetary relief.
4. **Dispute-risk model on the explanation-only population.** Use product, issue, state, and company features to flag closures most likely to generate a consumer dispute. A 10% reduction in disputes - 15,000 fewer adverse events over a comparable period.

### Data engineering
5. **Move product/issue taxonomy to a dimension table** so renames (like "Credit reporting" ? "Credit reporting, credit repair services, -") don't keep breaking downstream metrics.
6. **Migrate to Spark or Databricks** before the dataset doubles. 1.6 GB in pandas is already a fragile foundation for anything running on a schedule.

### Customer experience
7. **NLP triage on the ~30% of complaints with narratives.** Classify by severity and route high-risk cases to a remediation team instead of defaulting to explanation-only.

## 15. Roadmap

**Short-term (0-3 months)**
- Automate ETL from the CFPB public API ? cleaned warehouse table (daily incremental)
- Build a Power BI / Tableau dashboard for Compliance and CX teams
- Standardize company-name fuzzy matching to collapse 5,275 raw names into a clean entity table

**Medium-term (3-12 months)**
- Train and deploy a dispute-risk classifier on the explanation-only population
- Migrate from CSV + single-machine pandas to a Spark / Databricks pipeline
- Add NLP topic models on the consented narratives

**Long-term (12+ months)**
- Near-real-time intake monitoring with anomaly detection - catch Equifax-style spikes in hours, not months
- Federated benchmarking product for member institutions
- ML-driven resolution-quality scoring at company × product × geography

## 16. Closing summary

Took ~1.28 million rows of regulatory-grade complaint data, cleaned it, engineered the risk and operational features needed, and pulled out six insights that map directly to executive decisions: concentration, product-mix shift, channel digitization, response speed vs. depth, dispute risk, and tier-level operational gaps.

The single most actionable finding: **the system is fast but soft.** 97.5% of responses are timely, but only 18.6% deliver any form of relief - and 19.3% of resolved complaints draw a dispute. The next round of CX investment should target *resolution quality*, not response speed.

> One-sentence version: complaint volume in U.S. financial services is highly concentrated, increasingly digital, dominated by credit-reporting issues, and resolved fast but rarely remediated - making resolution quality the highest-ROI improvement target.

## 17. a) AI/ML Application - Dispute Risk Classifier

**Business Question:** Can we predict which complaints are likely to be disputed by the consumer before they happen?

**Why this matters:** Earlier in this analysis, we found that "Closed with 
explanation" looks like a quick, low-cost resolution, but about 1 in 5 of 
these consumers come back and dispute it. A dispute often means more time, more cost, and sometimes regulatory or legal escalation.

If we could predict which complaints are at high risk of being disputed *before* the company responds, we could route those cases to a more thorough review instead of a generic explanation, potentially preventing 
the dispute altogether.

**Approach:** We train a Random Forest classifier on the pre-April-2017 
population (the only period where the CFPB collected dispute outcomes), using complaint characteristics, product, issue, state, company size, and submission channel, to predict whether a complaint will be disputed.

In [28]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [45]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

In [36]:
# Dispute data only exists pre-April 2017 (CFPB stopped collecting it after)
df_ml = df[df['has_dispute_data'] == 1].copy()

print(f"Training population: {len(df_ml):,} complaints (pre-April 2017)")
print(f"Dispute rate in this population: {df_ml['is_disputed'].mean():.1%}")

Training population: 768,501 complaints (pre-April 2017)
Dispute rate in this population: 19.3%


In [37]:
# Target: did the consumer dispute the resolution?
y = df_ml['is_disputed']

# Features: characteristics of the complaint
features = ['Product', 'Issue', 'State', 'company_tier', 'Submitted via']
X = pd.get_dummies(df_ml[features], drop_first=True)

print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (768501, 179)


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set: {X_test.shape[0]:,} rows")

Training set: 614,800 rows
Test set: 153,701 rows


In [39]:
model = RandomForestClassifier(
    n_estimators=200,        # plus d'arbres
    max_depth=15,            # un peu plus profond
    min_samples_leaf=20,     # évite l'overfitting
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [40]:
y_pred = model.predict(X_test)

baseline_acc = 1 - y_test.mean()
model_acc = accuracy_score(y_test, y_pred)

print(f"{'='*50}")
print(f"Baseline (always predict 'no dispute'): {baseline_acc:.1%}")
print(f"Model accuracy: {model_acc:.1%}")
print(f"{'='*50}\n")

print(classification_report(y_test, y_pred, target_names=['No Dispute', 'Disputed']))

Baseline (always predict 'no dispute'): 80.7%
Model accuracy: 51.8%

              precision    recall  f1-score   support

  No Dispute       0.85      0.49      0.62    124025
    Disputed       0.23      0.64      0.34     29676

    accuracy                           0.52    153701
   macro avg       0.54      0.56      0.48    153701
weighted avg       0.73      0.52      0.57    153701



In [46]:
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"ROC-AUC Score: {auc:.3f}")

ROC-AUC Score: 0.588


**Interpreting these results:**

These categorical features alone provide modest signal. Since 30% of 
complaints include a written narrative from the consumer, we tested 
whether the text itself carries stronger predictive power.

In [42]:
# Filter to complaints WITH narrative
df_nlp = df[(df['has_dispute_data'] == 1) & (df['Consumer complaint narrative'].notna())].copy()
print(f"Rows with narrative: {len(df_nlp):,}")

Rows with narrative: 164,076


In [44]:
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
X_text = tfidf.fit_transform(df_nlp['Consumer complaint narrative'])

y_nlp = df_nlp['is_disputed']

In [47]:
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_nlp, test_size=0.2, random_state=42, stratify=y_nlp
)

# Logistic Regression works well with TF-IDF text features
text_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
text_model.fit(X_train_text, y_train_text)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [48]:
y_pred_text = text_model.predict(X_test_text)
auc_text = roc_auc_score(y_test_text, text_model.predict_proba(X_test_text)[:, 1])

print(f"ROC-AUC Score (text-based model): {auc_text:.3f}")
print(classification_report(y_test_text, y_pred_text, target_names=['No Dispute', 'Disputed']))

ROC-AUC Score (text-based model): 0.625
              precision    recall  f1-score   support

  No Dispute       0.84      0.60      0.70     25654
    Disputed       0.29      0.59      0.39      7162

    accuracy                           0.60     32816
   macro avg       0.57      0.59      0.55     32816
weighted avg       0.72      0.60      0.63     32816



**Final interpretation:** Adding the consumer's written narrative improved 
the model's discriminative power from an AUC of 0.59 to 0.63, confirming 
that the language consumers use carries real signal about dispute likelihood, 
beyond just the complaint category or company size.

Both models remain modest in absolute terms. This is expected: disputing a 
resolution is a complex consumer decision shaped by factors outside this 
dataset: financial stakes, personal urgency, and prior experience with the 
company are not captured here.


## 17. b) AI/ML Application 2 - Untimely Response Predictor

**Business Question:** Can we predict which complaints are at risk of missing 
the CFPB's 15-day response deadline?

**Why this matters:** We previously found that long-tail companies (Tier 3) 
miss the response deadline 9.5% of the time, compared to just ~1% for top-tier 
companies. If we can predict which complaints are at high risk of an untimely 
response *before* the deadline is missed, companies could prioritize those 
cases proactively.

**Approach:** We train a Random Forest classifier using the full dataset 
(not limited to pre-2017, since this field is always populated), using 
company tier, product, state, and submission channel as features.

In [50]:
# Target: was the response untimely?
y_untimely = df['is_untimely']

features_untimely = ['Product', 'State', 'company_tier', 'Submitted via']
X_untimely = pd.get_dummies(df[features_untimely], drop_first=True)

print(f"Feature matrix shape: {X_untimely.shape}")
print(f"Untimely rate: {y_untimely.mean():.1%}")

Feature matrix shape: (1282355, 87)
Untimely rate: 2.5%


In [51]:
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X_untimely, y_untimely, test_size=0.2, random_state=42, stratify=y_untimely
)

model_untimely = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    verbose=1
)
model_untimely.fit(X_train_u, y_train_u)

print("Model trained!")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   45.8s


Model trained!


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.6min finished


In [52]:
y_pred_u = model_untimely.predict(X_test_u)

baseline_acc_u = 1 - y_test_u.mean()
model_acc_u = accuracy_score(y_test_u, y_pred_u)
auc_u = roc_auc_score(y_test_u, model_untimely.predict_proba(X_test_u)[:, 1])

print(f"Baseline (always predict 'on time'): {baseline_acc_u:.1%}")
print(f"Model accuracy: {model_acc_u:.1%}")
print(f"ROC-AUC Score: {auc_u:.3f}")
print()
print(classification_report(y_test_u, y_pred_u, target_names=['On Time', 'Untimely']))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    1.4s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    2.5s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    1.0s


Baseline (always predict 'on time'): 97.5%
Model accuracy: 82.6%
ROC-AUC Score: 0.823

              precision    recall  f1-score   support

     On Time       0.99      0.83      0.90    250032
    Untimely       0.10      0.70      0.17      6439

    accuracy                           0.83    256471
   macro avg       0.54      0.76      0.54    256471
weighted avg       0.97      0.83      0.88    256471



[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    2.0s finished


**Interpreting these results:** Unlike the dispute classifier, this model 
shows strong discriminative power: a ROC-AUC of 0.82. While raw accuracy 
(82.6%) looks lower than the baseline (97.5%), that's expected given the 
class imbalance (only 2.5% of complaints are untimely): a model that always 
predicts "on time" would have high accuracy while catching zero late responses.

Our model correctly identifies 70% of complaints that will actually receive 
an untimely response : compared to 0% detection with no model at all.

**Why this model performs better than the dispute classifier:** Untimely 
responses are strongly driven by operational capacity, primarily company 
size (Tier 3 companies are untimely ~10x more often than Tier 1). This is a 
structural, predictable pattern, unlike consumer disputes which depend on 
subjective satisfaction that's harder to capture from complaint metadata alone.

**Practical takeaway:** A company could use this model to flag at-risk 
complaints in real time, proactively prioritizing cases likely to breach 
the 15-day deadline, rather than discovering the miss after the fact.